In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Install Vosk
!pip install vosk -q

import os
import json
import wave
import subprocess
from vosk import Model, KaldiRecognizer, SetLogLevel

# Suppress verbose debug logs
SetLogLevel(-1)

# 2. Define the input audio file
# IMPORTANT: Upload your own Greek audio file to the Colab files pane
# and change the filename below to match it exactly.
audio_file = "/content/GR_F_CeciliaC.mp3"

if not os.path.exists(audio_file):
    print(f"⚠️ Error: Could not find '{audio_file}'. Please upload your audio file to Colab.")
else:
    # 3. Convert the audio to 16kHz Mono 16-bit PCM for Vosk
    processed_file = "vosk_ready_greek.wav"
    print(f"Converting '{audio_file}' format via FFmpeg...")
    subprocess.run([
        "ffmpeg", "-y", "-i", audio_file,
        "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le",
        processed_file
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 4. Load the Pre-Trained Greek Model
    # Setting lang="el-gr" fetches the 1.1 GB Greek model.
    # This will take 1-2 minutes to download on Colab's network.
    print("Downloading and loading the Greek Vosk model (1.1 GB, please wait)...")
    model = Model(lang="el-gr")

    # 5. Run the Speech-to-Text Inference
    print("\n--- Starting Transcription ---\n")
    wf = wave.open(processed_file, "rb")
    recognizer = KaldiRecognizer(model, wf.getframerate())

    while True:
        # Read audio in small chunks
        data = wf.readframes(4000)
        if len(data) == 0:
            break

        # Process the waveform chunk
        if recognizer.AcceptWaveform(data):
            result = json.loads(recognizer.Result())
            if result.get('text'):
                print(f"Recognized: {result['text']}")

    # Catch any remaining words in the buffer
    final_result = json.loads(recognizer.FinalResult())
    if final_result.get('text'):
        print(f"Recognized: {final_result['text']}")

    print("\n--- Transcription Complete ---")

Converting '/content/GR_F_CeciliaC.mp3' format via FFmpeg...
lang el-gr does not exist


SystemExit: 1

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
